In [ ]:
import numpy as np
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import multiprocessing
%matplotlib widget

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport dejaq
%aimport dejaq.parallel
%aimport dejaq.queues
%aimport dejaq.remote
import dejaq

# Queue

In [ ]:
dq = dejaq.queues.DejaQueue(2000e6)
pdq = dejaq.queues.PicklableDejaQueue(2000e6)
mq = multiprocessing.Queue()

import time, hashlib
time.sleep(.2)
a = np.random.randint(0,255, (1, 100,1000,1000), dtype='uint8')

n = 10

hash = hashlib.md5(a).hexdigest()

def test(x, q):
    for i in range(n):
        q.put(x)
    for i in range(n):
        out = q.get()
        
def copytwice(x):
    for i in range(n):
        x.copy()
        x.copy()

%time copytwice(a)
%time test(a, dq)
%time test(a, pdq)
%time test(a, mq)


In [ ]:

dq2 = dejaq.queues.DejaQueue2(2000e6)
dejaq.queues.bench_queue(dq, np.array([1e6, 10e6, 100e6]).astype('int'), repeats=100)

In [ ]:
%load_ext line_profiler
%lprun -f dq.get test(a, dq)

In [ ]:
import pickle
pkl = pickle.dumps(mp.SimpleQueue())

In [ ]:
out = dq.get()

In [ ]:
out.head.value +=1

In [ ]:
# UCX one-way bandwidth microbenchmark (Jupyter-safe)
# pip install ucx-py ; ensure UCX is installed/loaded
import os, time, asyncio, gc
import numpy as np
import ucp
try:
    import nest_asyncio; nest_asyncio.apply()
except Exception:
    pass

# Uncomment to force transports as needed, e.g. CPU loopback:
# os.environ["UCX_TLS"] = "tcp,sm,self"
# os.environ["UCX_SOCKADDR_TLS_PRIORITY"] = "tcp"

SIZE_MB, ITERS, WARMUP = 5, 200, 20
server_done = asyncio.Event()

async def server_handler(ep: ucp.Endpoint):
    try:
        hdr = np.empty(2, dtype=np.uint64)       # [nbytes, total_iters]
        await ep.recv(hdr)
        nbytes, total = map(int, hdr)
        buf = np.empty(nbytes, dtype=np.uint8)
        for _ in range(total):
            await ep.recv(buf)
        await ep.send(np.array([1], dtype=np.uint8))  # ack
        await ep.close()
    finally:
        server_done.set()

async def main():
    listener = ucp.create_listener(server_handler, port=0)
    ep = await ucp.create_endpoint("127.0.0.1", listener.port)

    nbytes = SIZE_MB * 1024 * 1024
    total = WARMUP + ITERS
    await ep.send(np.array([nbytes, total], dtype=np.uint64))

    payload = np.empty(nbytes, dtype=np.uint8); payload[:] = 7  # touch pages

    for _ in range(WARMUP):
        await ep.send(payload)

    t0 = time.perf_counter()
    for _ in range(ITERS):
        await ep.send(payload)
    ack = np.empty(1, dtype=np.uint8)
    await ep.recv(ack)
    t1 = time.perf_counter()

    # orderly teardown to avoid UCXError in notebooks
    await ep.close()
    await server_done.wait()
    listener.close()

    gBps = (nbytes * ITERS) / (t1 - t0) / 1e9
    Gbps = 8 * gBps
    print(f"UCX one-way throughput: {gBps:.2f} GB/s ({Gbps:.2f} Gbit/s) "
          f"| msg={SIZE_MB} MB, iters={ITERS}, warmup={WARMUP}")

    # In notebooks, avoid reset; if you need it between runs, try:
    # await asyncio.sleep(0)
    # del ep, listener
    # gc.collect()
    # ucp.reset()

await main()


In [ ]:
a = np.random.randn(1,1000,1000)
dq.put(a)


In [ ]:
dq.view

In [ ]:
dq._write_buffer(a)


In [ ]:
%%time
for i in range(100):
    #dq.view[i:10_000_000+i] = a+i
    dq._write_buffer(a+i*1)

In [ ]:
dq.get() == Ellipsis

In [ ]:
for item in iter(dq):
    print(item.shape, item.mean())

In [ ]:
a = np.random.randn(1,1000,1000)
dq.put(a)
qa = dq.get()
a.mean() == qa.mean()

In [ ]:
import time

def timethis(fcn, *args):
    tic = time.time()
    fcn(*args)
    return time.time() - tic

out = []
nbytes = 10**np.r_[3:8.1:0.25]
for i in tqdm(nbytes):
    a = np.random.randint(0,255, (1,int(i)), dtype='uint8')
    out.append((timethis(copytwice, a), timethis(test, a, dq), timethis(test, a, mq)))

In [ ]:
fig, ax = plt.subplots(1,2, figsize = (10,4))
#plt.plot(nbytes, n*nbytes[:,None]/np.array(out)/1e9, label=['copy', 'dejaq', 'mp.Queue'])
ax[0].plot(nbytes, n*nbytes[:,None]/np.array(out)[:,1:]/1e9, '.', label=['DejaQueue', 'mp.Queue'])
ax[1].plot(nbytes, n*nbytes[:,None]/np.array(out)[:,1:]/1e9, '.', label=['DejaQueue', 'mp.Queue'])
ax[0].set(xscale = 'log', xlabel = 'item size (byte)', ylabel = 'transfer speed (GiB/s)', yscale = 'linear', ylim=(0,None))
ax[1].set(xscale = 'log', xlabel = 'item size (byte)', ylabel = 'transfer speed (GiB/s)', yscale = 'log')
ax[0].legend()
ax[1].legend()
plt.tight_layout()

In [ ]:
import numpy as np
from multiprocessing import Process
from dejaq import DejaQueue

def produce(queue):
    for i in range(100):
        random_shape = np.random.randint(5,10, size=3)
        array = np.random.randn(*random_shape)
        queue.put(dict(data=array, i=i))
        #print(f'produced {type(array)} {array.shape} {array.dtype}; meta: {i}; hash: {hash(array.tobytes())}\n', flush=True)
    print('put None')
    queue.put(None)

def consume(queue, pid):
    while True:
        d = queue.get()
        if d is None: return
        array, meta = d['data'], d['i']
        #print(f'consumer {pid} consumed {type(array)} {array.shape} {array.dtype}; meta: {meta}; hash: {hash(array.tobytes())}\n', flush=True)
    print('done')

queue = DejaQueue(bytes=10e6)
producer = Process(target=produce, args=(queue,))
consumers = [Process(target=consume, args=(queue, pid)) for pid in range(5)]
for c in consumers:
    c.start()
producer.start()
queue.queue.join()

In [ ]:
import numpy as np
from multiprocessing import Process
from dejaq import DejaQueue

def produce(queue):
    for i in range(10):
        arr = np.random.randn(100,200,300)
        data = dict(array=arr, i=i)
        queue.put(data)
        print(f'produced {type(arr)} {arr.shape} {arr.dtype}; meta: {i}; hash: {hash(arr.tobytes())}\n', flush=True)
    queue.put(None)
    print('producer done')

def consume(queue, pid):
    while True:
        data = queue.get()
        if data is None: break
        array, i = data['array'], data['i']
        print(f'consumer {pid} consumed {type(array)} {array.shape} {array.dtype}; index: {i}; hash: {hash(array.tobytes())}\n', flush=True)
    print(f'consumer {pid} done')

queue = DejaQueue(buffer_bytes=100e6)
producer = Process(target=produce, args=(queue,))
consumers = [Process(target=consume, args=(queue, pid)) for pid in range(3)]
for c in consumers:
    c.start()
producer.start()

# Parallel

In [ ]:
m = dejaq.parallel.lazymap(lambda n: sum((i*i) % 97 for i in range(5_000_000)), np.arange(100), n_workers=8)
np.array([i for i in tqdm(m)])

In [ ]:
mp.get_context("spawn")

In [ ]:
from dejaq import Parallel
from time import sleep
import numpy as np

@Parallel(n_workers=1)
class Producer:
    def __init__(self, arg1):
        self.arg1 = arg1
    def __call__(self, item):
        return item + self.arg1

@Parallel(n_workers=10)
class Processor:
    def __init__(self, arg1):
        self.arg1 = arg1
    def __call__(self, arg):
        sleep(0.1)
        return arg * self.arg1

@Parallel(n_workers=1)
class Consumer:
    def __init__(self, arg):
        self.arg1 = arg
    def __call__(self, arg):
        return arg + self.arg1

# stage0 = np.arange(100)
# stage1 = Producer(0.5)(stage0)
# stage2 = Processor(10.0)(stage1)
# stage3 = Consumer(1000)(stage2)

In [ ]:
from dejaq.parallel import lazymap
def test_lazymap_basic():
    f = lambda x: x * 2
    data = list(range(10))
    lm = lazymap(None, data, n_workers=1)
    result = list(lm)
test_lazymap_basic()

In [ ]:
import multiprocessing as mp
from multiprocessing.reduction import ForkingPickler
from dejaq.parallel import lazymap
lm = lazymap(None, range(10), n_workers=2)
#lm2 = lazymap(None, lm, n_workers=2)

In [ ]:
import time
import multiprocessing as mp
from multiprocessing.reduction import ForkingPickler
from dejaq.parallel import lazymap
def test_lazymap_ordered_output():
    def f(x):
        time.sleep(0.01 * (10 - x))  # out of order
        return x
    data = list(range(10))
    lm = lazymap(f, data, n_workers=3)
    result = list(lm)
    assert result == data, f"Output is not ordered: {result} != {data}"

test_lazymap_ordered_output()

In [ ]:
from dejaq.parallel import printer
import dejaq    
import multiprocessing as mp
mp.set_start_method("spawn", force=True)

dq = dejaq.DejaQueue(1e6)
c = mp.Condition()
v = mp.Value('i', 0)    


for pid in range(2):
    w = mp.Process(target=printer, args=(pid,dq, c, v, None, lm), kwargs=dict(dq=dq))
    w.start()

In [ ]:
import multiprocessing as mp

def worker(a):
    a[0] += 1

ctx = mp.get_context("spawn")            # or "fork" on Linux
arr = ctx.Array('i', 10, lock=False)     # shared-ctypes
p = ctx.Process(target=worker, args=(arr,))  # pass ONLY at start-up
p.start(); p.join()
print(arr[:]) 

In [ ]:
import multiprocessing as mp
import datetime




import threading
from collections import deque

def buffered_fanout(iterable, n, *, maxlen=1, overflow="drop_oldest"):
    """Fan out `iterable` into `n` generators with per-consumer buffers.

    Each output gets one copy of every input item. The source advances as the
    feeder thread pushes items into per-consumer deques.

    Args:
        iterable: Source iterable/iterator.
        n (int): Number of output generators (>= 2).
        maxlen (int): Per-consumer queue capacity.
            - If overflow == "block": when a queue reaches `maxlen`, the feeder
              **blocks** until that consumer catches up (classic back-pressure).
            - If overflow == "drop_oldest": deques drop the oldest items to make
              room automatically; slow consumers will miss frames.
            - Special case: `maxlen=1` and `overflow="block"` reproduces strict
              barrier semantics (the source can’t advance until all have taken
              the current item).
        overflow (str): "block" or "drop_oldest".

    Returns:
        tuple[generator, ...]: `n` generator objects.

    Notes:
        - Constant additional memory: O(n * maxlen).
        - If any consumer abandons without advancing and overflow="block",
          all producers/peers can stall. Use "drop_oldest" if you need liveness.
        - Exceptions from the source are propagated to all consumers on next poll.
    """
    if n < 2:
        raise ValueError("n must be >= 2")
    if overflow not in {"block", "drop_oldest"}:
        raise ValueError('overflow must be "block" or "drop_oldest"')
    if maxlen is not None and maxlen < 0:
        raise ValueError("maxlen must be >= 0 or None")

    it = iter(iterable)
    cond = threading.Condition()
    done = {"flag": False}
    error = {"exc": None}

    if overflow == "drop_oldest":
        # deque(maxlen) auto-drops oldest on append when full
        qs = [deque(maxlen=maxlen or None) for _ in range(n)]
    else:
        # We'll enforce capacity manually for blocking
        qs = [deque() for _ in range(n)]

    def feeder():
        try:
            for x in it:
                with cond:
                    if overflow == "block" and maxlen:
                        while any(len(q) >= maxlen for q in qs) and not done["flag"]:
                            cond.wait()
                    for q in qs:
                        q.append(x)
                    cond.notify_all()
        except BaseException as e:  # propagate StopIteration or other errors
            with cond:
                error["exc"] = e if not isinstance(e, StopIteration) else None
                done["flag"] = True
                cond.notify_all()
            return
        with cond:
            done["flag"] = True
            cond.notify_all()

    threading.Thread(target=feeder, daemon=True).start()

    def make_gen(i):
        q = qs[i]
        def gen():
            while True:
                with cond:
                    while not q and not done["flag"]:
                        cond.wait()
                    if q:
                        v = q.popleft()
                        if overflow == "block" and maxlen:
                            cond.notify_all()  # may unblock feeder
                    else:
                        # done and empty
                        if error["exc"]:
                            raise error["exc"]
                        return
                yield v
        return gen()

    return tuple(make_gen(i) for i in range(n))


In [ ]:
from dejaq.parallel import Ticker
ticker = Ticker()

In [ ]:
@Parallel(n_workers=1)
def test(item):
    sleep(.1)
    return item

@Parallel(n_workers=1)
def printer1(item):
    print(1, item)
    return 1, item

@Parallel(n_workers=1)
def printer2(item):
    print(2, item)
    return 2, item

@Parallel(n_workers=1)
def delay(item):
    sleep(.5)
    return item




ticker = Ticker()

dtt = test(ticker)
# bf0, bf1 = buffered_fanout(dtt, 2)
# fut1 = printer1(bf0).submit()
# fut2 = printer2(bf1).submit()
fut = printer2(dtt).submit()

In [ ]:
ticker.start()
sleep(.8)
ticker.pause()


In [ ]:
ticker.stop()

In [ ]:
fut

In [ ]:
import datetime
now_iter = iter(datetime.datetime.now, object())
for i in tqdm(now_iter):
    print(i)
    sleep(1)

In [ ]:
import dejaq, pickle, time

def test(arr):
    time.sleep(np.random.rand()/10)
    return arr + 1


In [ ]:
stage0 = range(10)
stage1 = dejaq.parallel.OrderedStage(test, n_workers=5)

In [ ]:
sm = dejaq.parallel.stage(test, n_workers=1).imap(range(10))

In [ ]:
list(sm)

In [ ]:
stage0 = range(10)
stage1 = Producer(0.5)(stage0)
stage2 = Processor(10.0)(stage1)
stage3 = Consumer(1000)(stage2)
out = stage3.compute()

In [ ]:
## class that wraps a python object, instantiates it in another process, allows you to call methods on it and set/get properties
import gc
import numpy as np
import dejaq
import multiprocessing as mp
import traceback, sys

class Actor:
    def __init__(self, cls, *args, **kwargs):
        self._cls = cls
        self._in_queue = dejaq.DejaQueue()
        self._out_queue = dejaq.DejaQueue()
        self._process = mp.Process(target=self._run, args=(cls, self._in_queue, self._out_queue, args, kwargs))
        self._process.start()
    @staticmethod
    def _run(cls, in_queue, out_queue, args, kwargs):
        def serialize_exception(e):
            exc_type, exc_value, exc_traceback = sys.exc_info()
            return {'type': exc_type.__name__, 'message': str(e), 'traceback': ''.join(traceback.format_tb(exc_traceback))}
        try:
            obj = cls(*args, **kwargs)
        except Exception as e:
            out_queue.put(dict(type='exception', exception=serialize_exception(e)))
            return
        while True:
            msg = in_queue.get()
            if msg is None: 
                del obj
                gc.collect()
                break
            try:
                if msg['type'] == 'method':
                    method = getattr(obj, msg['method'])
                    out = method(*msg['args'], **msg['kwargs'])
                    out_queue.put(out)
                elif msg['type'] == 'get':
                    out = getattr(obj, msg['attr'])
                    out_queue.put(out)
                elif msg['type'] == 'set':
                    setattr(obj, msg['attr'], msg['value'])
            except Exception as e:
                out_queue.put(dict(type='exception', exception=serialize_exception(e)))
    def _raise_remote_exception(self, exc_info):
        # Raise an exception received from the subprocess in the main process
        exc_type = type(exc_info['message'], (Exception,), {})
        raise exc_type(f"{exc_info['type']}: {exc_info['message']}\nTraceback:\n{exc_info['traceback']}")
    def __getattribute__(self, item):
        return super().__getattribute__(item)
    def __getattr__(self, attr):
        assert self._out_queue.empty(), "communication queue is not empty"
        def method_proxy(*args, **kwargs):
            self._in_queue.put(dict(type='method', method=attr, args=args, kwargs=kwargs))
            result = self._out_queue.get()
            if isinstance(result, dict) and result['type'] == 'exception':
                self._raise_remote_exception(result['exception'])
            return result
        self._in_queue.put(dict(type='get', attr=attr))
        result = self._out_queue.get()
        if isinstance(result, dict) and result['type'] == 'exception':
            self._raise_remote_exception(result['exception'])
        if callable(result):
            return method_proxy  # Return the proxy to call the method remotely
        return result
    def __setattr__(self, attr, value):
        if attr.startswith('_'):
            super().__setattr__(attr, value)
        else:
            assert self._in_queue.empty(), "communication queue is not empty"
            self._in_queue.put(dict(type='set', attr=attr, value=value))
    def __call__(self, *args, **kwargs):
        self._in_queue.put(dict(type='method', method='__call__', args=args, kwargs=kwargs))
        return self._out_queue.get()
    def __del__(self):
        self._in_queue.put(None)
        self._process.join()
    def __dir__(self):
        self._in_queue.put(dict(type='method', method='__dir__', args=(), kwargs={}))
        obj_attrs = self._out_queue.get() or []
        if isinstance(obj_attrs, dict) and obj_attrs['type'] == 'exception':
            self._raise_remote_exception(obj_attrs['exception'])
        return obj_attrs
    def __repr__(self):
        self._in_queue.put(dict(type='method', method='__repr__', args=[], kwargs={}))
        result = self._out_queue.get()
        if isinstance(result, dict) and result['type'] == 'exception':
            return "<Actor (error fetching repr)>"
        return f"<Actor wrapping: {result}>"

In [ ]:
import numpy as np
import time
from dejaq.parallel import Actor

class Foo:
    def __init__(self, par):
        self.par = par
    def bar(self, x):
        time.sleep(.5)
        return self.par

actor = Actor(Foo, 3)
#actor = Foo(3)

In [ ]:
res = actor.bar(1)
print(res)

In [ ]:
actor.par = 5

In [ ]:
fut = actor.bar.delayed(1)
print(fut)

In [ ]:
%%time
fut.get()

In [ ]:
print(fut.get())

In [ ]:
import numpy as np
np.random.randint(0, 2**63)

In [ ]:
s = 'sdfsdf2234'


In [ ]:
# fast_ipc.py  — Windows + Unix, high-speed IPC Queue & Condition
import sys, time, struct
import numpy as np
from multiprocessing import shared_memory

# ---------- Cross-platform named sync wrappers ----------
if sys.platform.startswith("win"):
    import win32event, win32con, win32api

    class NamedMutex:
        def __init__(self, name: str):
            self._h = win32event.CreateMutex(None, False, name)
        def acquire(self, timeout=None) -> bool:
            ms = win32con.INFINITE if timeout is None else int(timeout*1000)
            rc = win32event.WaitForSingleObject(self._h, ms)
            return rc in (win32con.WAIT_OBJECT_0, win32con.WAIT_ABANDONED)
        def release(self): win32event.ReleaseMutex(self._h)
        def close(self):   win32api.CloseHandle(self._h)

    class NamedSemaphore:
        def __init__(self, name: str, initial: int, maximum: int = 2**31-1):
            self._h = win32event.CreateSemaphore(None, initial, maximum, name)
        def acquire(self, timeout=None) -> bool:
            ms = win32con.INFINITE if timeout is None else int(timeout*1000)
            rc = win32event.WaitForSingleObject(self._h, ms)
            return rc == win32con.WAIT_OBJECT_0
        def release(self, n: int = 1):
            for _ in range(n): win32event.ReleaseSemaphore(self._h, 1, None)
        def close(self): win32api.CloseHandle(self._h)

else:
    import posix_ipc

    class NamedMutex:
        def __init__(self, name: str):
            if not name.startswith("/"): name = "/" + name
            try:
                self._sem = posix_ipc.Semaphore(name, flags=posix_ipc.O_CREAT|posix_ipc.O_EXCL, initial_value=1)
                self._owner = True
            except posix_ipc.ExistentialError:
                self._sem = posix_ipc.Semaphore(name); self._owner = False
        def acquire(self, timeout=None) -> bool:
            if timeout is None: self._sem.acquire(); return True
            end = time.monotonic() + timeout
            while True:
                try:
                    self._sem.acquire(timeout=0); return True
                except posix_ipc.BusyError:
                    if time.monotonic() >= end: return False
                    time.sleep(0.0005)
        def release(self): self._sem.release()
        def close(self):   self._sem.close()

    class NamedSemaphore:
        def __init__(self, name: str, initial: int):
            if not name.startswith("/"): name = "/" + name
            try:
                self._sem = posix_ipc.Semaphore(name, flags=posix_ipc.O_CREAT|posix_ipc.O_EXCL, initial_value=initial)
            except posix_ipc.ExistentialError:
                self._sem = posix_ipc.Semaphore(name)
        def acquire(self, timeout=None) -> bool:
            if timeout is None: self._sem.acquire(); return True
            end = time.monotonic() + timeout
            while True:
                try:
                    self._sem.acquire(timeout=0); return True
                except posix_ipc.BusyError:
                    if time.monotonic() >= end: return False
                    time.sleep(0.0005)
        def release(self, n: int = 1):
            for _ in range(n): self._sem.release()
        def close(self):   self._sem.close()

# ---------- Fast named bounded queue (fixed slot size) ----------
class FastNamedQueue:
    """
    Bounded MPMC queue of bytes/pickled blobs.
    SharedMemory layout:
      header (32 bytes): [head:int64, tail:int64, cap:int64, slot:int64]
      data: cap * slot bytes; each slot starts with uint32 payload length.
    """
    HDR_FMT = "<qqqq"            # head, tail, cap, slot
    HDR_SIZE = struct.calcsize(HDR_FMT)
    LEN_SIZE = 4                 # uint32 length header per slot

    def __init__(self, name: str, capacity: int, slot_bytes: int, create: bool = True):
        self.name = name
        self.mutex = NamedMutex(name + "_mtx")
        self.items = NamedSemaphore(name + "_it", 0)                # #filled slots
        self.slots = NamedSemaphore(name + "_sl", capacity)         # #free slots
        total = self.HDR_SIZE + capacity * slot_bytes
        if create:
            shm = shared_memory.SharedMemory(name=name, create=True, size=total)
            self.shm = shm; self.buf = shm.buf
            hdr = struct.pack(self.HDR_FMT, 0, 0, capacity, slot_bytes)
            self.buf[:self.HDR_SIZE] = hdr
        else:
            shm = shared_memory.SharedMemory(name=name)
            self.shm = shm; self.buf = shm.buf
        # views
        self._hdr_mv = self.buf[:self.HDR_SIZE]
        self._data_mv = self.buf[self.HDR_SIZE:]

    # ---- internals ----
    def _read_hdr(self):
        return list(struct.unpack(self.HDR_FMT, self._hdr_mv))
    def _write_hdr(self, head, tail, cap, slot):
        self._hdr_mv[:] = struct.pack(self.HDR_FMT, head, tail, cap, slot)

    # ---- API ----
    def put(self, data: bytes, timeout: float | None = None):
        if not isinstance(data, (bytes, bytearray, memoryview)):
            # caller can pickle externally for max speed
            import pickle
            data = pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL)
        cap, slot = self._read_hdr()[2:4]
        if len(data) + self.LEN_SIZE > slot:
            raise ValueError(f"payload {len(data)} exceeds slot {slot - self.LEN_SIZE}")
        if not self.slots.acquire(timeout): raise TimeoutError("put timeout")
        try:
            if not self.mutex.acquire(timeout): raise TimeoutError("put mutex timeout")
            head, tail, cap, slot = self._read_hdr()
            pos = tail
            tail = (tail + 1) % cap
            self._write_hdr(head, tail, cap, slot)
        finally:
            self.mutex.release()
        # write payload into slot
        base = pos * slot
        mv = self._data_mv[base: base + slot]
        struct.pack_into("<I", mv, 0, len(data))
        mv[self.LEN_SIZE:self.LEN_SIZE + len(data)] = data
        self.items.release(1)

    def get(self, timeout: float | None = None) -> bytes:
        if not self.items.acquire(timeout): raise TimeoutError("get timeout")
        try:
            if not self.mutex.acquire(timeout): raise TimeoutError("get mutex timeout")
            head, tail, cap, slot = self._read_hdr()
            pos = head
            head = (head + 1) % cap
            self._write_hdr(head, tail, cap, slot)
        finally:
            self.mutex.release()
        # read payload
        base = pos * slot
        mv = self._data_mv[base: base + slot]
        (n,) = struct.unpack_from("<I", mv, 0)
        out = bytes(mv[self.LEN_SIZE: self.LEN_SIZE + n])
        self.slots.release(1)
        return out

    def close(self):
        self.shm.close(); self.mutex.close(); self.items.close(); self.slots.close()

# ---------- Fast named condition variable ----------
class FastNamedCondition:
    """
    Condition variable built from a named mutex + named semaphore + SharedMemory header.
    Usage:
        cond = FastNamedCondition("condX")
        # waiter:
        cond.acquire()
        ok = cond.wait(timeout=5.0)   # releases & re-acquires the same mutex
        # ... proceed ...
        cond.release()
        # notifier:
        cond.acquire(); cond.notify(n=1); cond.release()
    """
    HDR_FMT = "<qq"    # waiters:int64, seq:int64
    HDR_SIZE = struct.calcsize(HDR_FMT)

    def __init__(self, name: str, create: bool = True):
        self.name = name
        self.mutex = NamedMutex(name + "_mtx")   # external mutex for users
        self._sem  = NamedSemaphore(name + "_sem", 0)
        if create:
            shm = shared_memory.SharedMemory(name=name + "_hdr", create=True, size=self.HDR_SIZE)
            self.shm = shm; self.buf = shm.buf
            self.buf[:] = struct.pack(self.HDR_FMT, 0, 0)
        else:
            shm = shared_memory.SharedMemory(name=name + "_hdr")
            self.shm = shm; self.buf = shm.buf

    def _read(self):
        return list(struct.unpack(self.HDR_FMT, self.buf[:self.HDR_SIZE]))
    def _write(self, waiters, seq):
        self.buf[:self.HDR_SIZE] = struct.pack(self.HDR_FMT, waiters, seq)

    # Expose the mutex so callers can use `with cond.locked():`
    def acquire(self, timeout=None): return self.mutex.acquire(timeout)
    def release(self): self.mutex.release()

    def wait(self, timeout: float | None = None) -> bool:
        """Assumes caller holds the mutex (like threading.Condition)."""
        w, s = self._read()
        self._write(w+1, s)
        self.mutex.release()
        ok = self._sem.acquire(timeout)
        self.mutex.acquire()
        # decrement waiters under the lock
        w2, s2 = self._read()
        self._write(max(0, w2-1), s2)
        return ok

    def notify(self, n: int = 1):
        """Assumes caller holds the mutex. Wakes up to n waiters."""
        w, s = self._read()
        k = min(n, w)
        if k:
            self._write(w, s + k)
            self._sem.release(k)

    def notify_all(self): self.notify(n=10**9)

    def close(self): self.shm.close(); self.mutex.close(); self._sem.close()


In [ ]:
# Queue throughput microbenchmark (Jupyter cell)
import os, time, multiprocessing as mp, numpy as np

try:
    from fast_ipc import FastNamedQueue  # if you saved the class in a module
except ImportError:
    FastNamedQueue  # already in namespace from prior cell

# ---- parameters ----
PRODUCERS = 2
ITEMS_PER_PRODUCER = 50_000
PAYLOAD_BYTES = 1024          # bytes per message (will be pickled if not bytes)
CAPACITY = 8192               # ring slots; keep >> producers to avoid blocking
NAME = f"fq_{os.getpid()}_{int(time.time())}"
SLOT_BYTES = PAYLOAD_BYTES + 8   # 4B length + small headroom

ctx = mp.get_context("spawn")

def producer(name, capacity, slot_bytes, n_items, payload, start_evt):
    q = FastNamedQueue(name, capacity, slot_bytes, create=False)
    start_evt.wait()
    for _ in range(n_items):
        q.put(payload)
    q.close()

def consumer(name, capacity, slot_bytes, total_items, start_evt, outq):
    q = FastNamedQueue(name, capacity, slot_bytes, create=False)
    start_evt.wait()
    t0 = time.perf_counter()
    total_bytes = 0
    for _ in range(total_items):
        b = q.get()
        total_bytes += len(b)
    t1 = time.perf_counter()
    q.close()
    outq.put((t1 - t0, total_bytes))

if __name__ == "__main__":
    # creator
    q = FastNamedQueue(NAME, CAPACITY, SLOT_BYTES, create=True)
    # fixed payload
    payload = bytes(np.random.randint(0, 256, size=PAYLOAD_BYTES, dtype=np.uint8))

    start_evt = ctx.Event()
    outq = ctx.Queue()

    procs = []
    for _ in range(PRODUCERS):
        p = ctx.Process(target=producer, args=(NAME, CAPACITY, SLOT_BYTES, ITEMS_PER_PRODUCER, payload, start_evt))
        p.start(); procs.append(p)
    total_items = PRODUCERS * ITEMS_PER_PRODUCER
    c = ctx.Process(target=consumer, args=(NAME, CAPACITY, SLOT_BYTES, total_items, start_evt, outq))
    c.start()

    # start!
    time.sleep(0.05)
    start_evt.set()

    # gather
    for p in procs: p.join()
    c.join()
    elapsed, total_bytes = outq.get()

    q.close()  # close creator's handle

    ops = total_items / elapsed
    gBps = total_bytes / elapsed / 1e9
    print(f"Producers={PRODUCERS}, items/producer={ITEMS_PER_PRODUCER}, payload={PAYLOAD_BYTES} B")
    print(f"Elapsed={elapsed:.3f} s | ops/s={ops:,.0f} | throughput={gBps:.3f} GB/s")


In [ ]:
fq = FastNamedQueue('fnqname', 100, int(10e6), create=True)

In [ ]:
fq.put(b'hello')

In [ ]:
fq.get()

In [ ]:
import time, statistics

def run_once_named(ctx, cond, tmo=0.5):
    pa, pb = ctx.Pipe()
    p = ctx.Process(target=dejaq.queues.child_named, args=(cond.base, pb, tmo)); p.start()
    pa.recv()  # child has the mutex
    # Wait until child registered as waiter
    while int(cond._state[0]) == 0:
        time.sleep(0.0005)
    with cond:
        cond.notify(1)
    ok, dt = pa.recv(); p.join()
    return ok, dt

def run_once_mp(ctx, tmo=0.5):
    lock = mp.Lock()
    cond = mp.Condition(lock)
    pa, pb = ctx.Pipe()
    p = ctx.Process(target=dejaq.queues.child_mp, args=(cond, lock, pb, tmo)); p.start()
    pa.recv()
    with lock:
        cond.notify(1)
    ok, dt = pa.recv(); p.join()
    return ok, dt

def main(trials=20):
    ctx = mp.get_context("spawn")  # macOS uses 'spawn'
    named = dejaq.queues.NamedCondition(create=True)
    named_times = []
    for _ in range(trials):
        ok, dt = run_once_named(ctx, named, 0.5)
        assert ok, "named notify(1) failed"
        named_times.append(dt)
    # timeout sanity check
    pa, pb = ctx.Pipe()
    p = ctx.Process(target=dejaq.queues.child_named, args=(named.base, pb, 0.3)); p.start()
    pa.recv()
    while int(named._state[0]) == 0: time.sleep(0.0005)
    ok_to, dt_to = pa.recv(); p.join()
    print(f"NamedCondition timeout: ok={ok_to} dt≈{dt_to:.1f} µs (expect False)")

    mp_times = []
    for _ in range(trials):
        ok, dt = run_once_mp(ctx, 0.5)
        assert ok, "mp notify(1) failed"
        mp_times.append(dt)

    def stats(xs):
        return dict(median=statistics.median(xs),
                    p90=statistics.quantiles(xs, n=10)[8],
                    min=min(xs), max=max(xs))
    print("NamedCondition notify(1):", stats(named_times))
    print("mp.Condition notify(1)  :", stats(mp_times))

    named.close(); named.unlink()

if __name__ == "__main__":
    main()

In [ ]:
import pickle, queue
_ = pickle.dumps(queue.SimpleQueue())

In [ ]:
# remote.py
from __future__ import annotations
import os, time, uuid, traceback, multiprocessing as mp
from dataclasses import dataclass
from typing import Any, Callable, Optional, Dict, Tuple

# Transport: your picklable, zero-copy queue (mp.Queue-like API: raises TimeoutError on timeout)
from dejaq.queues import PicklableDejaQueue


# ============================== Wire types & errors ==============================

@dataclass
class _Req:
    """Request envelope.

    Attributes:
        call_id: Correlation id for matching replies.
        reply: Name/base of the caller's mailbox queue.
        kind: One of {"call", "apply", "ping", "shutdown"}.
        name: Method name (actor) or unused (remote func).
        args: Positional arguments.
        kwargs: Keyword arguments.
    """
    call_id: str
    reply: str
    kind: str
    name: str
    args: tuple
    kwargs: dict


@dataclass
class _Rep:
    """Reply envelope.

    Attributes:
        call_id: Correlation id (matches the request).
        ok: True if the call succeeded; False if it raised.
        payload: Result object (if ok), else a tuple (etype, eargs, tb_str).
    """
    call_id: str
    ok: bool
    payload: Any


class RemoteError(RuntimeError):
    """Exception raised locally for errors that occurred in a remote worker/actor."""
    def __init__(self, etype: str, eargs: Tuple[Any, ...], remote_tb: str):
        super().__init__(f"{etype}{eargs}\n-- Remote traceback --\n{remote_tb}")
        self.remote_type = etype
        self.remote_args = eargs
        self.remote_traceback = remote_tb


# ============================== Server/worker loops ==============================

def _actor_server(cls: type, ctor_args: tuple, ctor_kwargs: dict, req_name: str) -> None:
    """Actor loop: instantiates `cls` and services _Req from a request queue.

    The server replies to the mailbox specified by each request's `reply`.
    Designed to be module-level for Windows 'spawn'.
    """
    req = PicklableDejaQueue(name=req_name, create=False)
    obj = cls(*ctor_args, **ctor_kwargs)

    rep_cache: Dict[str, PicklableDejaQueue] = {}
    def repq(name: str) -> PicklableDejaQueue:
        q = rep_cache.get(name)
        if q is None:
            q = PicklableDejaQueue(name=name, create=False)
            rep_cache[name] = q
        return q

    while True:
        msg: _Req = req.get()  # blocking
        if msg.kind == "shutdown":
            repq(msg.reply).put(_Rep(msg.call_id, True, None))
            break
        try:
            if msg.kind == "ping":
                out = "pong"
            elif msg.kind == "call":
                out = getattr(obj, msg.name)(*msg.args, **msg.kwargs)
            else:
                raise ValueError(f"unknown kind {msg.kind!r}")
            repq(msg.reply).put(_Rep(msg.call_id, True, out))
        except BaseException as e:
            repq(msg.reply).put(
                _Rep(msg.call_id, False, (type(e).__name__, e.args, traceback.format_exc()))
            )


def _func_worker(fn_ser: Tuple[str, str] | Callable, req_name: str) -> None:
    """Function worker loop: applies a function for 'apply' requests.

    Args:
      fn_ser: Either a callable or a (module, name) pair for lazy import in the worker.
      req_name: Name/base of the shared request queue.
    """
    if callable(fn_ser):
        fn = fn_ser
    else:
        mod, name = fn_ser
        fn = getattr(__import__(mod, fromlist=[name]), name)

    req = PicklableDejaQueue(name=req_name, create=False)
    rep_cache: Dict[str, PicklableDejaQueue] = {}
    def repq(name: str) -> PicklableDejaQueue:
        q = rep_cache.get(name)
        if q is None:
            q = PicklableDejaQueue(name=name, create=False)
            rep_cache[name] = q
        return q

    while True:
        msg: _Req = req.get()
        if msg.kind == "shutdown":
            repq(msg.reply).put(_Rep(msg.call_id, True, None))
            break
        try:
            if msg.kind != "apply":
                raise ValueError(f"unknown kind {msg.kind!r}")
            out = fn(*msg.args, **msg.kwargs)
            repq(msg.reply).put(_Rep(msg.call_id, True, out))
        except BaseException as e:
            repq(msg.reply).put(
                _Rep(msg.call_id, False, (type(e).__name__, e.args, traceback.format_exc()))
            )


# ============================== Client-side demux & futures ==============================

class _Mailbox:
    """Demultiplex replies by `call_id` for a single client process.

    Keeps a small buffer of out-of-order replies so multiple Futures can be awaited
    concurrently. Each *process* gets its own mailbox queue; servers reply to the
    specific mailbox requested → no cross-process races.

    Args:
      rep_q: The reply/mailbox queue owned by this process.
    """
    def __init__(self, rep_q: PicklableDejaQueue) -> None:
        self.q = rep_q
        self._buf: Dict[str, _Rep] = {}

    def wait(self, call_id: str, timeout: Optional[float]) -> _Rep:
        """Wait for the reply with matching `call_id`.

        Args:
          call_id: Correlation id to wait for.
          timeout: Seconds (float) or None for blocking.

        Returns:
          The matching _Rep.

        Raises:
          TimeoutError: If the deadline elapses before the reply arrives.
        """
        if call_id in self._buf:
            return self._buf.pop(call_id)
        deadline = None if timeout is None else (time.time() + timeout)
        while True:
            rem = None if deadline is None else max(0.0, deadline - time.time())
            rep: _Rep = self.q.get(timeout=rem)  # raises TimeoutError on deadline
            if rep.call_id == call_id:
                return rep
            self._buf[rep.call_id] = rep


class Future:
    """Handle for an outstanding remote call."""

    def __init__(self, mbox: _Mailbox, call_id: str) -> None:
        self._mbox, self._id = mbox, call_id

    def result(self, timeout: Optional[float] = None):
        """Return the remote result or raise RemoteError."""
        rep = self._mbox.wait(self._id, timeout)
        if rep.ok:
            return rep.payload
        et, ea, tb = rep.payload
        raise RemoteError(et, ea, tb)


# ============================== Public API: Actor & RemoteFunc ==============================

class Actor:
    """Run a class instance in a separate process and call its methods remotely.

    Each client process owns a *mailbox* queue. Every request carries that mailbox
    name and a unique `call_id`. Replies are sent to that mailbox; the local demux
    buffers out-of-order replies so multiple Futures can be awaited safely.

    Example:
        >>> class Counter:
        ...     def __init__(self, x=0): self.x = x
        ...     def inc(self, n=1): self.x += n; return self.x
        >>>
        >>> a = Actor(Counter, 10)
        >>> a.inc(5)            # 15
        >>> fut = a.inc_async(7)
        >>> fut.result()        # 22
        >>> a.close()

    Args:
      cls: Class to run in the actor process (must be importable top-level).
      *args: Positional args for the class constructor.
      buffer_bytes: Size of each queue (request/mailbox) in bytes.
      start_method: Multiprocessing start method (default 'spawn' for portability).
      **kwargs: Keyword args for the class constructor.
    """
    def __init__(self, cls: type, *args,
                 buffer_bytes: int = 8_000_000,
                 start_method: str = "spawn",
                 **kwargs) -> None:
        base = f"act-{os.getpid()}-{uuid.uuid4().hex[:8]}"
        self._rep = PicklableDejaQueue(buffer_bytes=buffer_bytes, name=base+"_mb", create=True)   # mailbox
        self._mbox = _Mailbox(self._rep)
        self._req = PicklableDejaQueue(buffer_bytes=buffer_bytes, name=base+"_req", create=True)  # requests
        ctx = mp.get_context(start_method)
        self._p = ctx.Process(target=_actor_server, args=(cls, args, kwargs, self._req.base))
        self._p.start()
        self._closed = False

    # --- internals ---
    def _send(self, kind: str, name: str, args: tuple, kwargs: dict) -> str:
        cid = uuid.uuid4().hex
        self._req.put(_Req(cid, self._rep.base, kind, name, args, kwargs))
        return cid

    # --- dynamic method proxy ---
    def __getattr__(self, name: str):
        """Dynamic proxy: `actor.method(x, y, timeout=...)` and `actor.method_async(...)`."""
        if name.endswith("_async"):
            real = name[:-6]
            def _async(*args, **kwargs):
                cid = self._send("call", real, args, kwargs)
                return Future(self._mbox, cid)
            return _async

        def _sync(*args, timeout: Optional[float] = None, **kwargs):
            cid = self._send("call", name, args, kwargs)
            rep = self._mbox.wait(cid, timeout)
            if rep.ok:
                return rep.payload
            et, ea, tb = rep.payload
            raise RemoteError(et, ea, tb)
        return _sync

    def ping(self) -> str:
        """Liveness probe; returns 'pong' if the actor is responsive."""
        cid = self._send("ping", "", (), {})
        return self._mbox.wait(cid, 2.0).payload

    def close(self, timeout: float = 2.0) -> None:
        """Gracefully stop the actor process."""
        if self._closed: return
        try:
            cid = self._send("shutdown", "", (), {})
            _ = self._mbox.wait(cid, timeout)
        except Exception:
            pass
        self._p.join(timeout)
        if self._p.is_alive():
            self._p.terminate(); self._p.join()
        self._closed = True

    # context manager sugar
    def __enter__(self): return self
    def __exit__(self, exc_type, exc, tb): self.close()


class RemoteFunc:
    """Run a function in N worker processes (stateless pool).

    Each call is a request on a shared work queue; each *client process* has its
    own mailbox queue for replies (no cross-process demux races).

    Example:
        >>> def f(x): return x * x
        >>> rf = RemoteFunc(f, workers=4)
        >>> rf(12)               # 144
        >>> rf.map(range(5))     # [0, 1, 4, 9, 16]
        >>> fut = rf.submit(7); fut.result()  # 49
        >>> rf.close()

    Args:
      fn: Callable to execute in workers (importable top-level preferred).
      workers: Number of worker processes.
      buffer_bytes: Size of each queue (request/mailbox) in bytes.
      start_method: Multiprocessing start method (default 'spawn').
    """
    def __init__(self, fn: Callable, workers: int = 1,
                 buffer_bytes: int = 8_000_000,
                 start_method: str = "spawn") -> None:
        base = f"rf-{os.getpid()}-{uuid.uuid4().hex[:8]}"
        self._rep = PicklableDejaQueue(buffer_bytes=buffer_bytes, name=base+"_mb", create=True)    # mailbox
        self._mbox = _Mailbox(self._rep)
        self._req = PicklableDejaQueue(buffer_bytes=buffer_bytes, name=base+"_req", create=True)   # work queue
        ctx = mp.get_context(start_method)

        # Prefer (module, name) reference for spawn-friendliness; fall back to pickled callable.
        try:
            mod, name = fn.__module__, fn.__name__
            getattr(__import__(mod, fromlist=[name]), name)  # validate importability
            fn_ref: Tuple[str, str] | Callable = (mod, name)
        except Exception:
            fn_ref = fn

        self._ps = [ctx.Process(target=_func_worker, args=(fn_ref, self._req.base))
                    for _ in range(int(workers))]
        for p in self._ps: p.start()

    # --- API ---
    def __call__(self, *args, timeout: Optional[float] = None, **kwargs):
        """Apply fn(*args, **kwargs) and block for the result."""
        cid = uuid.uuid4().hex
        self._req.put(_Req(cid, self._rep.base, "apply", "", args, kwargs))
        rep = self._mbox.wait(cid, timeout)
        if rep.ok:
            return rep.payload
        et, ea, tb = rep.payload
        raise RemoteError(et, ea, tb)

    def submit(self, *args, **kwargs) -> Future:
        """Submit fn(*args, **kwargs) asynchronously; returns a Future."""
        cid = uuid.uuid4().hex
        self._req.put(_Req(cid, self._rep.base, "apply", "", args, kwargs))
        return Future(self._mbox, cid)

    def map(self, iterable, *, timeout: Optional[float] = None):
        """Map fn over an iterable; returns a list of results (preserves order)."""
        cids = []
        for x in iterable:
            cid = uuid.uuid4().hex
            self._req.put(_Req(cid, self._rep.base, "apply", "", (x,), {}))
            cids.append(cid)
        out = []
        for cid in cids:
            rep = self._mbox.wait(cid, timeout)
            if rep.ok:
                out.append(rep.payload)
            else:
                et, ea, tb = rep.payload
                raise RemoteError(et, ea, tb)
        return out

    def close(self, timeout: float = 2.0) -> None:
        """Gracefully stop all workers."""
        for _ in self._ps:
            cid = uuid.uuid4().hex
            self._req.put(_Req(cid, self._rep.base, "shutdown", "", (), {}))
        deadline = time.time() + timeout
        for p in self._ps:
            rem = max(0.0, deadline - time.time())
            p.join(rem)
            if p.is_alive():
                p.terminate(); p.join()

    # context manager sugar
    def __enter__(self): return self
    def __exit__(self, exc_type, exc, tb): self.close()


In [ ]:
from dejaq.remote import Actor, RemoteError

class Counter:
    def __init__(self, start=0): self.x = start
    def inc(self, n=1): self.x += n; return self.x
    def get(self): return self.x

import multiprocessing as mp
mp.set_start_method("spawn", force=True)

a = Actor(Counter, 10)         # actor owns a Counter(start=10)
print(a.ping())                   # -> 'pong'
print(a.inc(5))                   # -> 15
fut = a.inc_async(7)              # asynchronous call
print(fut.result())               # -> 22
print(a.get())                    # -> 22

In [ ]:
a.ping()

In [ ]:
from dejaq.queues import DejaQueue, PicklableDejaQueue
import numpy as np
from tqdm.auto import tqdm

dq = PicklableDejaQueue(100e6)


In [ ]:
from dejaq.queues import DejaQueue, PicklableDejaQueue
import numpy as np
from tqdm.auto import tqdm

dq = PicklableDejaQueue(100e6)

frame = np.random.randint(0, 256, (1, 10, 1000,1000), dtype=np.uint8).ravel().view('B')

for i in tqdm(range(100)):
    for i in range(5):
        dq.put(frame)
    for i in range(5):
        _ = dq.get()

# Remote

In [ ]:
from dejaq.remote import Actor, ActorDecorator

In [ ]:
@ActorDecorator
class Test:
    def __init__(self):
        self.x = 3
    def f(self, a):
        return self.x + a

In [ ]:
test = Test()

In [ ]:
%%time
test.ping()


In [ ]:
%load_ext line_profiler

In [ ]:
%lprun -f test._send test.ping(_ensure_open=False)